# OpenTelemetry for Agent Telemetry

**OpenTelemetry (OTel)** is a vendor-neutral standard for **traces**, **metrics**, and **logs**. Agent platforms (Langfuse, LangSmith, Datadog, Grafana Tempo) ingest **OTLP** — OpenTelemetry Protocol — so you instrument once and export anywhere.

```
  ShopCo / ReleaseFlow agent
         │
         ▼
  ┌──────────────────┐
  │  OTel Tracer     │  start/end spans
  └────────┬─────────┘
           ▼
  ┌──────────────────┐
  │  Span processor  │  batch, enrich attrs
  └────────┬─────────┘
           ▼
  ┌──────────────────┐
  │  OTLP Exporter   │  → collector → backend
  └──────────────────┘
```

This notebook implements an **OTel-compatible span model** in plain Python, instruments **ShopCo Support Hub** and **ReleaseFlow**, applies **GenAI semantic attributes**, and exports trace JSON in OTLP-like shape — no collector setup required.

In [ ]:
"""
OpenTelemetry for agent telemetry — ShopCo + ReleaseFlow.
"""

import json
import os
import re
import time
import uuid
from contextlib import contextmanager
from contextvars import ContextVar
from dataclasses import dataclass, field
from datetime import datetime, timezone
from enum import Enum
from typing import Any, Iterator

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-proj-placeholder-replace-with-your-real-key")
os.environ.setdefault("OPENAI_API_KEY", OPENAI_API_KEY)

USE_LIVE_LLM = False
USE_OTEL_SDK = False  # set True if opentelemetry-api is installed


def pretty(obj: Any) -> str:
    return json.dumps(obj, indent=2, default=str)


def utc_ns() -> int:
    return int(datetime.now(timezone.utc).timestamp() * 1_000_000_000)


POLICY_SNIPPETS = {
    "returns": "Returns within 30 days. [pol-returns-01]",
    "shipping": "Free shipping over $50. [pol-shipping-02]",
}

ORDERS_DB = {
    "ORD-1001": {"order_id": "ORD-1001", "status": "shipped"},
    "ORD-1002": {"order_id": "ORD-1002", "status": "processing"},
}

RELEASE_CHECKS = {"unit_tests": "pass", "security_scan": "warn"}

print("OTel lab ready.")

---

## 1. OTel Core Concepts

| Concept | Meaning |
|---------|--------|
| **Trace** | One end-to-end agent run (`trace_id`) |
| **Span** | One operation (LLM call, tool, router) |
| **Parent span** | Nesting — tool spans under agent span |
| **Attributes** | Key-value metadata (`gen_ai.request.model`) |
| **Events** | Timestamped notes on a span (tool observation) |
| **Context propagation** | Pass `trace_id` across services / agents |

---

## 2. Span and Status Model

In [ ]:
class SpanStatus(str, Enum):
    UNSET = "UNSET"
    OK = "OK"
    ERROR = "ERROR"


@dataclass
class SpanEvent:
    name: str
    timestamp_ns: int
    attributes: dict[str, Any] = field(default_factory=dict)


@dataclass
class OtlpSpan:
    trace_id: str
    span_id: str
    parent_span_id: str | None
    name: str
    kind: str
    start_time_unix_nano: int
    end_time_unix_nano: int | None = None
    attributes: dict[str, Any] = field(default_factory=dict)
    events: list[SpanEvent] = field(default_factory=list)
    status: SpanStatus = SpanStatus.UNSET

    def duration_ms(self) -> float:
        if self.end_time_unix_nano is None:
            return 0.0
        return (self.end_time_unix_nano - self.start_time_unix_nano) / 1_000_000


def new_trace_id() -> str:
    return uuid.uuid4().hex


def new_span_id() -> str:
    return uuid.uuid4().hex[:16]


print("Span model ready.")

---

## 3. Context Propagation — Active Span Stack

In [ ]:
_current_span: ContextVar[OtlpSpan | None] = ContextVar("current_span", default=None)
_baggage: ContextVar[dict[str, str]] = ContextVar("baggage", default={})


class InMemorySpanExporter:
    def __init__(self) -> None:
        self.spans: list[OtlpSpan] = []

    def export(self, spans: list[OtlpSpan]) -> None:
        self.spans.extend(spans)

    def clear(self) -> None:
        self.spans.clear()


class Tracer:
    def __init__(self, name: str, exporter: InMemorySpanExporter) -> None:
        self.name = name
        self.exporter = exporter
        self._trace_id: str | None = None

    @contextmanager
    def start_as_current_span(self, name: str, kind: str = "INTERNAL", **attrs: Any) -> Iterator[OtlpSpan]:
        parent = _current_span.get()
        if parent is None and self._trace_id is None:
            self._trace_id = new_trace_id()
        trace_id = self._trace_id or (parent.trace_id if parent else new_trace_id())
        span = OtlpSpan(
            trace_id=trace_id,
            span_id=new_span_id(),
            parent_span_id=parent.span_id if parent else None,
            name=name,
            kind=kind,
            start_time_unix_nano=utc_ns(),
            attributes=dict(attrs),
        )
        bag = _baggage.get()
        if bag:
            span.attributes.update({f"baggage.{k}": v for k, v in bag.items()})
        token = _current_span.set(span)
        try:
            yield span
            span.status = SpanStatus.OK
        except Exception as exc:
            span.status = SpanStatus.ERROR
            span.attributes["error.message"] = str(exc)
            raise
        finally:
            span.end_time_unix_nano = utc_ns()
            self.exporter.export([span])
            _current_span.reset(token)


exporter = InMemorySpanExporter()
tracer = Tracer("shopco.support", exporter)
print("Tracer ready.")

---

## 4. GenAI Semantic Conventions (Attributes)

Emerging conventions for LLM spans (names may evolve — check OTel GenAI semconv):

| Attribute | Example |
|-----------|--------|
| `gen_ai.system` | `openai` |
| `gen_ai.request.model` | `gpt-4o-mini` |
| `gen_ai.usage.input_tokens` | `120` |
| `gen_ai.usage.output_tokens` | `45` |
| `gen_ai.operation.name` | `chat` |
| `tool.name` | `get_order` |

In [ ]:
def llm_span_attrs(model: str, input_tokens: int, output_tokens: int) -> dict[str, Any]:
    return {
        "gen_ai.system": "openai",
        "gen_ai.request.model": model,
        "gen_ai.operation.name": "chat",
        "gen_ai.usage.input_tokens": input_tokens,
        "gen_ai.usage.output_tokens": output_tokens,
    }


def tool_span_attrs(tool_name: str, **extra: Any) -> dict[str, Any]:
    return {"tool.name": tool_name, "span.kind": "CLIENT", **extra}


print(llm_span_attrs("gpt-4o-mini", 100, 30))

---

## 5. ShopCo Agent — OTel Instrumented

In [ ]:
def get_order(order_id: str) -> dict[str, Any]:
    return ORDERS_DB.get(order_id.upper(), {"status": "not_found"})


def search_policy(query: str) -> str:
    return POLICY_SNIPPETS["returns"] if "return" in query.lower() else POLICY_SNIPPETS["shipping"]


def shopco_agent_otel(message: str, session_id: str) -> str:
    _baggage.set({"session.id": session_id})
    exporter.clear()
    t = Tracer("shopco.support", exporter)

    with t.start_as_current_span("agent.run", kind="SERVER", user_message=message[:100]):
        with t.start_as_current_span("router.classify", kind="INTERNAL"):
            intent = "return" if "return" in message.lower() else "order"
            time.sleep(0.001)

        parts: list[str] = []
        m = re.search(r"ORD-[0-9]{4}", message.upper())
        if m:
            with t.start_as_current_span("tool.get_order", kind="CLIENT", **tool_span_attrs("get_order", order_id=m.group(0))):
                rec = get_order(m.group(0))
                _current_span.get().events.append(SpanEvent("tool.result", utc_ns(), {"order.status": rec["status"]}))
                parts.append(f"Status: {rec['status']}")

        if intent == "return":
            with t.start_as_current_span("llm.compose", kind="CLIENT", **llm_span_attrs("gpt-4o-mini", 90, 40)):
                with t.start_as_current_span("tool.search_policy", kind="CLIENT", **tool_span_attrs("search_policy")):
                    pol = search_policy(message)
                    parts.append(pol)

        return " ".join(parts) if parts else "How can we help?"


answer = shopco_agent_otel("Can I return ORD-1001?", session_id="sess-42")
print("Answer:", answer)
print("Spans exported:", len(exporter.spans))

---

## 6. Inspect Exported Spans — Parent/Child Tree

In [ ]:
def span_tree(spans: list[OtlpSpan]) -> list[str]:
    by_id = {s.span_id: s for s in spans}
    lines: list[str] = []
    for s in spans:
        indent = "  " if s.parent_span_id else ""
        if s.parent_span_id and s.parent_span_id in by_id:
            parent = by_id[s.parent_span_id]
            depth = 1 + sum(1 for x in spans if x.span_id == parent.parent_span_id)
            indent = "  " * (2 if s.parent_span_id else 0)
        ms = round(s.duration_ms(), 2)
        lines.append(f"{indent}{s.name} [{s.status.value}] {ms}ms")
    return lines


for line in span_tree(exporter.spans):
    print(line)

---

## 7. OTLP-Style JSON Export

In [ ]:
def to_otlp_json(spans: list[OtlpSpan], service_name: str) -> dict[str, Any]:
    return {
        "resourceSpans": [{
            "resource": {"attributes": [{"key": "service.name", "value": {"stringValue": service_name}}]},
            "scopeSpans": [{
                "scope": {"name": "agent-telemetry-lab"},
                "spans": [
                    {
                        "traceId": s.trace_id,
                        "spanId": s.span_id,
                        "parentSpanId": s.parent_span_id or "",
                        "name": s.name,
                        "kind": s.kind,
                        "startTimeUnixNano": str(s.start_time_unix_nano),
                        "endTimeUnixNano": str(s.end_time_unix_nano or ""),
                        "attributes": [{"key": k, "value": {"stringValue": str(v)}} for k, v in s.attributes.items()],
                        "status": {"code": s.status.value},
                    }
                    for s in spans
                ],
            }],
        }],
    }


otlp_preview = to_otlp_json(exporter.spans, "shopco-support")
print("Span count:", len(exporter.spans))
print(pretty(otlp_preview["resourceSpans"][0]["scopeSpans"][0]["spans"][0]))

---

## 8. ReleaseFlow — Pipeline Spans

In [ ]:
def releaseflow_otel(version: str) -> dict[str, Any]:
    rf_exporter = InMemorySpanExporter()
    t = Tracer("releaseflow", rf_exporter)
    gate = "proceed"

    with t.start_as_current_span("release.pipeline", kind="SERVER", "release.version": version):
        for check, status in RELEASE_CHECKS.items():
            with t.start_as_current_span(f"ci.{check}", kind="CLIENT", **tool_span_attrs(check, status=status)):
                if status == "fail":
                    gate = "block"
        with t.start_as_current_span("gate.decision", kind="INTERNAL", gate=gate):
            pass

    return {"version": version, "gate": gate, "spans": len(rf_exporter.spans)}


print(releaseflow_otel("2.4.0"))

---

## 9. Error Spans — Tool Not Found

In [ ]:
err_exporter = InMemorySpanExporter()
err_tracer = Tracer("shopco.support", err_exporter)

try:
    with err_tracer.start_as_current_span("tool.get_order", **tool_span_attrs("get_order")):
        raise ValueError("Invalid order_id format ORD-XYZ")
except ValueError:
    pass

err_span = err_exporter.spans[0]
print("Status:", err_span.status.value)
print("Error attr:", err_span.attributes.get("error.message"))

---

## 10. Aggregate Metrics from Spans

In [ ]:
def aggregate_trace_metrics(spans: list[OtlpSpan]) -> dict[str, Any]:
    llm_spans = [s for s in spans if s.name.startswith("llm.")]
    tool_spans = [s for s in spans if s.name.startswith("tool.")]
    tokens_in = sum(s.attributes.get("gen_ai.usage.input_tokens", 0) for s in llm_spans)
    tokens_out = sum(s.attributes.get("gen_ai.usage.output_tokens", 0) for s in llm_spans)
    return {
        "trace_id": spans[0].trace_id if spans else None,
        "span_count": len(spans),
        "llm_calls": len(llm_spans),
        "tool_calls": len(tool_spans),
        "tokens_in": tokens_in,
        "tokens_out": tokens_out,
        "total_duration_ms": round(sum(s.duration_ms() for s in spans), 2),
        "error_spans": sum(1 for s in spans if s.status == SpanStatus.ERROR),
    }


print(pretty(aggregate_trace_metrics(exporter.spans)))

---

## 11. Cross-Service Propagation (Concept)

When ShopCo **handoffs** to a policy microservice, inject W3C `traceparent` header so child service spans share `trace_id`:

```
traceparent: 00-{trace_id}-{parent_span_id}-01
```

Multi-agent systems need **one trace per user goal**, not one trace per agent.

In [ ]:
def format_traceparent(span: OtlpSpan) -> str:
    return f"00-{span.trace_id}-{span.span_id}-01"


root = exporter.spans[0] if exporter.spans else None
if root:
    print("traceparent:", format_traceparent(root))

---

## 12. Where OTLP Goes in Production

| Component | Role |
|-----------|------|
| **OTel SDK** | Create spans in your agent code |
| **OTel Collector** | Receive OTLP, filter, route |
| **Backend** | Langfuse, Jaeger, Tempo, Datadog |

This lab's `InMemorySpanExporter` stands in for `OTLPSpanExporter(endpoint=...)`.

---

## 13. Optional — Real OpenTelemetry SDK

Set `USE_OTEL_SDK = True` after installing `opentelemetry-api` and `opentelemetry-sdk`.

In [ ]:
def try_otel_sdk() -> dict[str, Any]:
    try:
        from opentelemetry import trace
        from opentelemetry.sdk.trace import TracerProvider
        from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter as SdkExporter
    except ImportError:
        return {"status": "otel_not_installed", "hint": "pip install opentelemetry-api opentelemetry-sdk"}

    provider = TracerProvider()
    sdk_exporter = SdkExporter()
    from opentelemetry.sdk.trace.export import SimpleSpanProcessor
    provider.add_span_processor(SimpleSpanProcessor(sdk_exporter))
    trace.set_tracer_provider(provider)
    tr = trace.get_tracer("shopco")
    with tr.start_as_current_span("sdk.demo"):
        pass
    return {"status": "ok", "spans": len(sdk_exporter.get_finished_spans())}


if USE_OTEL_SDK:
    print(try_otel_sdk())
else:
    print("Simulation mode — InMemorySpanExporter used above.")

---

## 14. Optional Live LLM with Span Attributes

Set `USE_LIVE_LLM = True` to record token usage on a real call.

In [ ]:
def live_llm_with_attrs() -> dict[str, Any]:
    from langchain_core.messages import HumanMessage
    from langchain_openai import ChatOpenAI

    llm = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY, temperature=0)
    resp = llm.invoke([HumanMessage(content="One line ShopCo return policy.")])
    meta = getattr(resp, "usage_metadata", None) or {}
    return llm_span_attrs(
        "gpt-4o-mini",
        meta.get("input_tokens", 0),
        meta.get("output_tokens", 0),
    )


if USE_LIVE_LLM:
    print(live_llm_with_attrs())
else:
    print("Offline attrs:", llm_span_attrs("gpt-4o-mini", 90, 40))

---

## 15. Common Mistakes

| Mistake | Symptom | Fix |
|---------|---------|-----|
| One span per HTTP only | Blind to tools | Child spans per LLM/tool |
| Missing `trace_id` on handoff | Broken distributed trace | W3C traceparent |
| No GenAI attributes | Can't sum tokens in backend | `gen_ai.usage.*` |
| Span per message not session | Fragmented sessions | Baggage `session.id` |
| Vendor-only SDK | Lock-in | OTel OTLP export |

---

## 16. Quiz

1. What is the difference between a trace and a span?
2. What does OTLP stand for?
3. Name two GenAI semantic attributes.
4. Why use baggage for `session.id`?
5. What span status indicates tool failure?

---

## 17. Summary

**OpenTelemetry** standardizes agent telemetry: nested spans, GenAI attributes, OTLP export, and cross-service context. ShopCo and ReleaseFlow demos showed tracer instrumentation, events, error status, and OTLP JSON shape.

Instrument **every LLM and tool call** as a child span, attach **token attributes**, propagate **trace context** across agents, and export to your collector of choice.